## Alcance del notebook
- Fuente de datos: `data/processed/companies_clean.csv` generada por el ETL.
- Columnas utilizadas: `empresa` y `region_casa_matriz`.
- Objetivo: ofrecer lecturas claras sobre cuántas empresas están en cada región y quiénes son.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data/processed/companies_clean.csv"
pio.templates.default = "plotly_dark"

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data/processed/companies_clean.csv"

In [3]:
companies = pd.read_csv(DATA_PATH, usecols=["empresa", "region_casa_matriz"])
companies.sort_values("region_casa_matriz").head()

,empresa,region_casa_matriz
73,Vidal Y Urrutia Limitada,Biobío
24,Eitech Spa,Biobío
27,Ewood Spa,Biobío
60,Rootman Spa,Biobío
30,Floresencia Limitada,Biobío


### Distribución de empresas por región
La gráfica siguiente sirve para conversaciones ejecutivas: muestra el volumen por región, ordenado de menor a mayor para identificar polos con rapidez.

In [4]:
region_counts = (
    companies.groupby("region_casa_matriz")
    .size()
    .reset_index(name="numero_empresas")
    .sort_values(by="numero_empresas", ascending=True)
)

fig = px.bar(
    region_counts,
    x="numero_empresas",
    y="region_casa_matriz",
    orientation="h",
    text="numero_empresas",
    color="numero_empresas",
    color_continuous_scale=px.colors.sequential.Teal,
)
fig.update_layout(
    title="Concentración de empresas por región",
    xaxis_title="Número de empresas",
    yaxis_title="Región",
    coloraxis_showscale=False,
)
fig.update_traces(textposition="outside")
fig.show()

### Quién es quién dentro de cada región
El treemap permite explorar qué empresas están presentes dentro de cada región sin agregar otras columnas; es ideal para sesiones de scouting o revisión de portafolios.

In [5]:
treemap_df = companies.copy()
treemap_df["empresa_count"] = 1

fig = px.treemap(
    treemap_df,
    path=["region_casa_matriz", "empresa"],
    values="empresa_count",
    color="region_casa_matriz",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig.update_layout(title="Empresas por región (treemap)")
fig.update_traces(textinfo="label+value")
fig.show()